## 医疗分诊 DistilBERT-LoRA 微调训练.ipynb

### 项目说明
本 Notebook 用于：医疗症状紧急度四分类 LoRA 微调
模型：DistilBERT（轻量编码器，CPU 可训练）
任务：emergency / urgent / routine / self_care 四级分诊
特点：轻量、快速、泛化性强、适配医疗安全分诊前置拦截逻辑

### 1. 安装依赖（首次运行执行）

In [ ]:
# 如需全新环境，解开注释执行
# !pip install torch transformers datasets peft scikit-learn numpy

In [1]:
import env_config

### 2. 导入全局依赖 + 配置

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import torch
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)

# 全局模型与路径配置
BASE_MODEL = "distilbert-base-uncased"   #轻量化蒸馏版英文 BERT；uncased = 不区分大小写，专门做英文文本分类
OUT_DIR = "./artifacts/triage-lora" #模型保存路径

# 四分类标签定义（与数据集脚本完全对齐）
LABELS = ["emergency", "urgent", "routine", "self_care"]
LABEL2ID = {label: i for i, label in enumerate(LABELS)}
ID2LABEL = {i: label for label, i in LABEL2ID.items()} #文字标签转为神经网络能识别的数字

print("✅ 全局配置加载完成")
print("标签映射：", LABEL2ID)
print("是否可用GPU：", torch.cuda.is_available())

✅ 全局配置加载完成
标签映射： {'emergency': 0, 'urgent': 1, 'routine': 2, 'self_care': 3}
是否可用GPU： True


### 3. 数据集加载/自动生成函数

In [3]:
def build_dataset(out_dir: str = "finetuning/data"):
    """复用数据集构建逻辑，内置在Notebook中保证独立可运行"""
    import random
    random.seed(42)

    SYMPTOMS = {
        "emergency": [
            "crushing chest pain radiating to my left arm",
            "sudden difficulty breathing and I'm gasping for air",
            "one side of my face is drooping and my speech is slurred",
            "I'm bleeding heavily and it won't stop after ten minutes",
            "I passed out and just regained consciousness confused",
            "my lips and throat are swelling up and I can barely swallow after eating peanuts",
            "the worst headache of my life that came on all of a sudden",
            "a sudden severe headache along with confusion and blurred vision",
            "a headache that started after hitting my head hard and I feel drowsy",
            "I'm vomiting blood",
            "I have thoughts of ending my life right now",
            "my child swallowed a bottle of pills",
            "severe abdominal pain and I can't stand up straight",
            "I think I'm having a heart attack",
            "sudden weakness on one side of my body",
            "I was in a car accident and my leg looks deformed",
            "a seizure that has lasted more than five minutes",
        ],
        "urgent": [
            "a fever of 103 that has lasted three days",
            "I twisted my ankle and it's swelling fast, maybe broken",
            "persistent vomiting and diarrhea and I feel very dehydrated",
            "a deep cut on my hand that probably needs stitches",
            "an eye injury after something splashed into it",
            "a dog bit me and broke the skin",
            "worsening pain from what I think is a kidney stone",
            "a rash that's spreading fast with a fever",
            "I ran out of my insulin and my blood sugar readings are very high",
            "my asthma inhaler isn't helping and I'm still wheezing",
            "a urinary tract infection with fever and back pain",
            "chest tightness that started after exercise but isn't severe",
            "sudden severe ear pain with hearing loss",
        ],
        "routine": [
            "mild joint stiffness in the mornings for the past few weeks",
            "I'd like to schedule my annual checkup",
            "a small skin rash that isn't spreading or itching much",
            "occasional mild acid reflux after meals",
            "I want to discuss adjusting my blood pressure medication",
            "recurring mild headaches a couple times a month",
            "migraines with light sensitivity that happen every few weeks and respond to my usual medication",
            "a throbbing headache with nausea and light sensitivity that I get occasionally",
            "a bad headache and I am sensitive to light, but I get this every month before my period",
            "a bad headache with sensitivity to light that lines up with my usual migraine pattern",
            "a mole I'd like a doctor to take a look at eventually",
            "ongoing lower back stiffness after sitting all day",
            "I'd like a referral for a routine eye exam",
            "mild seasonal allergies that come back every spring",
            "I want to ask about starting a new exercise routine safely",
            "follow-up on my cholesterol test results from last month",
        ],
        "self_care": [
            "a runny nose and mild sore throat, feels like a common cold",
            "a small paper cut on my finger",
            "mild muscle soreness after a workout yesterday",
            "a slight headache after a long day at the computer",
            "a headache with mild light sensitivity that gets better after resting in a dark room",
            "a bad headache and I am sensitive to light, but resting in a quiet dark room usually helps",
            "a bad headache with sensitivity to light after a stressful day, nothing like before",
            "a minor sunburn on my shoulders",
            "occasional mild heartburn after spicy food",
            "a little bit of dry, itchy skin in winter",
            "mild hiccups that started an hour ago",
            "a stuffy nose from seasonal pollen",
            "slight fatigue after a poor night's sleep",
            "a small bruise from bumping into a table",
            "mild constipation after traveling",
        ],
    }

    TEMPLATES_TRAIN = [
        "I have {s}.",
        "I'm dealing with {s}.",
        "I've been experiencing {s}.",
        "For the past hour I've had {s}.",
        "My symptom is {s}.",
        "Lately I've noticed {s}.",
        "I woke up with {s}.",
        "Right now I have {s}.",
    ]

    TEMPLATES_TEST = [
        "Is it serious that I have {s}?",
        "What should I do about {s}?",
        "I'm worried because I have {s}.",
        "Should I see someone about {s}?",
        "Just started having {s}, any advice?",
    ]

    def _generate(templates: list[str]) -> list[dict]:
        rows = []
        for label, phrases in SYMPTOMS.items():
            for phrase in phrases:
                for template in templates:
                    rows.append({"text": template.format(s=phrase), "label": label})
        random.shuffle(rows)
        return rows

    train_rows = _generate(TEMPLATES_TRAIN)
    test_rows = _generate(TEMPLATES_TEST)

    path = Path(out_dir)
    path.mkdir(parents=True, exist_ok=True)
    (path / "train.jsonl").write_text("\n".join(json.dumps(r) for r in train_rows))
    (path / "test.jsonl").write_text("\n".join(json.dumps(r) for r in test_rows))
    return train_rows, test_rows


def _load_or_build_rows():
    """加载本地数据集，不存在则自动生成"""
    train_path = Path("finetuning/data/train.jsonl")
    test_path = Path("finetuning/data/test.jsonl")
    if train_path.exists() and test_path.exists():
        train_rows = [json.loads(l) for l in train_path.read_text().splitlines() if l]
        test_rows = [json.loads(l) for l in test_path.read_text().splitlines() if l]
    else:
        train_rows, test_rows = build_dataset()
    return train_rows, test_rows

# 执行加载/生成
train_rows, test_rows = _load_or_build_rows()
print(f"✅ 数据集加载完成 | 训练集：{len(train_rows)} 条 | 测试集：{len(test_rows)} 条")


✅ 数据集加载完成 | 训练集：488 条 | 测试集：305 条


### 4. 数据集预处理 & Tokenize

 Dataset.from_list 普通列表转为 Hugging‑Face Dataset，Trainer、map 函数都必须依靠该类型。

In [4]:
def _to_hf_dataset(rows: list[dict], tokenizer) -> Dataset:
    """转换为HuggingFace标准数据集 + 批量编码"""
    ds = Dataset.from_list([
        {"text": r["text"], "label": LABEL2ID[r["label"]]} 
        for r in rows
    ])
    # 短句固定64长度，截断truncation=True + 补全 padding="max_length"
    ds = ds.map(
        lambda ex: tokenizer(ex["text"], truncation=True, padding="max_length", max_length=64),
        batched=True
    )
    return ds

# 加载分词器
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
train_ds = _to_hf_dataset(train_rows, tokenizer)
test_ds = _to_hf_dataset(test_rows, tokenizer)

print("✅ 数据预处理完成")
print("单条样本结构：", train_ds[0].keys())
print("示例文本：", train_ds[0]["text"])
print("对应标签ID：", train_ds[0]["label"])


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/488 [00:00<?, ? examples/s]

Map:   0%|          | 0/305 [00:00<?, ? examples/s]

✅ 数据预处理完成
单条样本结构： dict_keys(['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'])
示例文本： I'm dealing with mild muscle soreness after a workout yesterday.
对应标签ID： 3


### 5. 评估指标函数（Acc / Precision / Recall / F1）

In [5]:
def compute_metrics(eval_pred):
    """医疗分类任务宏平均评估，均衡四类指标"""
    logits, labels = eval_pred  #logits为模型预测结果，labels为真实标签
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )  
    #average="macro"  宏平均：先分别算出 4 个分类各自的精确率、召回率，之后直接取平均值  
    #zero_division=0  当某个类别没有任何预测样本时，防止除数为零报错，直接赋值指标为 0
    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

print("✅ 评估指标函数初始化完成")


✅ 评估指标函数初始化完成


### 6. 加载模型 + LoRA 配置（核心微调）

In [6]:
# 加载DistilBERT分类底座模型
## Distil‑BERT 主干 + 内置的分类全连接层 (分类头)，专门用来做文本分类
base_model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, 
    num_labels=len(LABELS), 
    id2label=ID2LABEL, 
    label2id=LABEL2ID
)

# DistilBERT专属LoRA配置（q_lin / v_lin 注意力层）
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,   # 序列分类任务
    r=8,                          # LoRA秩
    lora_alpha=16,                # 缩放系数  r*2
    lora_dropout=0.1,             # 防过拟合
    target_modules=["q_lin", "v_lin"]
)

# 注入LoRA适配器
model = get_peft_model(base_model, lora_config) #把 LoRA 外挂矩阵注入到指定注意力层
model.print_trainable_parameters()

print("✅ LoRA模型加载完成")


model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 741,124 || all params: 67,697,672 || trainable%: 1.0948
✅ LoRA模型加载完成


### 7. 训练超参数配置

In [7]:
args = TrainingArguments(
    output_dir="finetuning/checkpoints",
    num_train_epochs=6,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-4,
    eval_strategy="epoch",    # 每轮结束评估
    save_strategy="no",       # 不保存中间checkpoint
    logging_steps=10,
    report_to=[],             # 关闭wandb日志
    use_cpu=not torch.cuda.is_available(), # 自动适配CPU/GPU
)

print("✅ 训练超参配置完成")
print("当前训练设备：", "CPU" if args.use_cpu else "GPU")


✅ 训练超参配置完成
当前训练设备： GPU


### 8. 启动训练 + 评估 + 保存模型

Training Loss（训练集损失）    模型在已经见过的训练数据集上面的犯错代价
Validation Loss（验证集损失）   模型在从来没有见过的测试数据集上面的犯错代价，用来检验模型能不能举一反三、泛化能力。

In [8]:
# 初始化训练器
trainer = Trainer(
    model=model, 
    args=args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics, 
)

# 开始训练
trainer.train()

# 最终测试集评估
final_metrics = trainer.evaluate()
print("\n🎯 最终测试集指标：")
print(final_metrics)

# 保存LoRA权重、分词器、标签映射
#parents=True：多级文件夹不存在就逐层创建
# exist_ok=True：文件夹已经存在也不会抛出报错
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
model.save_pretrained(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)
Path(OUT_DIR, "label_map.json").write_text(json.dumps({"id2label": ID2LABEL, "label2id": LABEL2ID}))

print(f"\n✅ 模型保存成功！路径：{OUT_DIR}")


[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,1.254668,1.199230,0.531148,0.528170,0.491385,0.356200
2,0.647358,0.583413,0.862295,0.868818,0.862783,0.862052
3,0.305605,0.329742,0.927869,0.928813,0.926263,0.926749
4,0.181437,0.241313,0.944262,0.945143,0.943906,0.943213
5,0.102476,0.210566,0.924590,0.930458,0.920950,0.922838
6,0.090696,0.164713,0.950820,0.951997,0.948884,0.949444


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.090696,0.164713,6,0.950820,0.951997,0.948884,0.949444



🎯 最终测试集指标：
{'eval_loss': 0.16471326351165771, 'eval_accuracy': 0.9508196721311475, 'eval_precision': 0.9519967701820864, 'eval_recall': 0.9488838612368025, 'eval_f1': 0.9494442478216516}

✅ 模型保存成功！路径：finetuning/artifacts/triage-lora


### 9. 单样本推理测试（验证模型效果）

关键点：LoRA 不会修改基础模型原生配置，每次加载基础模型都必须手动告知它现在是 4 分类任务。

In [11]:
from peft import PeftModel, PeftConfig

# 加载保存好的LoRA模型
peft_config = PeftConfig.from_pretrained(OUT_DIR)
infer_model = AutoModelForSequenceClassification.from_pretrained(
    peft_config.base_model_name_or_path,
    num_labels=len(LABELS)) # 加载base模型
infer_model = PeftModel.from_pretrained(infer_model, OUT_DIR)  #绑定lora矩阵
infer_model.eval()

# 测试样例（急症+常规+自理）
test_cases = [
    "I have crushing chest pain radiating to my left arm",
    "I have mild headache every month before my period",
    "I have a slight sore throat and runny nose"
]

print("🔍 模型推理测试：\n")
for text in test_cases:
    inputs = tokenizer(text, truncation=True, padding="max_length", max_length=64, return_tensors="pt")
    with torch.no_grad():
        logits = infer_model(**inputs).logits
        pred_id = int(torch.argmax(logits, dim=-1)[0])
        pred_label = ID2LABEL[pred_id]
    print(f"输入：{text}")
    print(f"分诊结果：{pred_label}\n")


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


🔍 模型推理测试：

输入：I have crushing chest pain radiating to my left arm
分诊结果：emergency

输入：I have mild headache every month before my period
分诊结果：routine

输入：I have a slight sore throat and runny nose
分诊结果：self_care

